# 05 — GPU worker for the web application

Runs a Celery worker on the **gpu** queue (docs/platform_plan.md, phase 3). Jobs started from
the web application's batch page with "Colab GPU (with narratives)" wait in that queue until
this notebook is running, then run here: stage 1 and SHAP, the classifier, and Llama 3.1 8B
narratives for inconsistent verdicts. Progress appears on the batch page.

**Before the first run**, add two Colab Secrets (key icon in the left bar), both reachable
from the internet:

| Secret | Value |
|---|---|
| `DATABASE_URL` | the hosted PostgreSQL the web application uses, `postgres://...` |
| `REDIS_URL` | the hosted Redis, `rediss://...` |

and `GITHUB_TOKEN` as for the other notebooks. Keep this tab open while jobs run: Colab ends
idle sessions and caps free sessions at 12 hours. The last cell runs until you stop it;
a job that is cut off returns to the queue (late acknowledgement) for the next worker.


In [ ]:
BRANCH = 'django'
CONCURRENCY = 1   # one job at a time: the classifier and vLLM share the T4


## 1. Bootstrap: Drive, repository, secrets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys
REPO_DIR = '/content/claimguard'

DRIVE_ENV = '/content/drive/MyDrive/claimguard/.env'

def load_env_file(path):
    # Minimal .env reader (KEY=value lines), so this cell needs no extra package.
    if not os.path.exists(path):
        return
    for line in open(path):
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            key, value = line.split('=', 1)
            os.environ.setdefault(key.strip(), value.strip().strip('"\''))

def github_token():
    # The kernel runs on Colab's VM, so a local .env (VS Code's jupyter.envFile) never
    # reaches it. Read the .env kept in Drive, then a Colab secret, then prompt.
    load_env_file(DRIVE_ENV)
    if os.getenv('GITHUB_TOKEN'):
        return os.environ['GITHUB_TOKEN']
    try:
        from google.colab import userdata
        return userdata.get('GITHUB_TOKEN')
    except Exception:
        pass
    from getpass import getpass
    return getpass('GITHUB_TOKEN (read-only, not saved): ').strip()

if not os.path.exists(REPO_DIR):
    token = github_token()
    if not token:
        raise RuntimeError('No GITHUB_TOKEN: add a Colab secret or enter it at the prompt')
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH,
                    f'https://{token}@github.com/victorgpt0/claimguard.git', REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '-q'], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
COMMIT = subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()

from google.colab import userdata
for key in ('DATABASE_URL', 'REDIS_URL'):
    value = os.environ.get(key) or userdata.get(key)
    if not value:
        raise RuntimeError(f'Add the Colab Secret {key} first (see the top of this notebook)')
    os.environ[key] = value
# Paths and thresholds for Colab (Drive under /content/drive/MyDrive/claimguard).
os.environ['CLAIMGUARD_CONFIG'] = f'{REPO_DIR}/config/config.example.yaml'
os.environ['DJANGO_DEBUG'] = 'false'
os.environ.setdefault('DJANGO_SECRET_KEY', 'worker-only-not-serving-pages')
print('commit', COMMIT)


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
# The ML packages as notebook 03 runs them on Colab's CUDA torch, then the web and worker
# packages pinned from uv.lock (requirements/colab-worker.txt).
!pip install -q vllm==0.19.0 peft shap
!pip install -q -r requirements/colab-worker.txt
# Colab preinstalls torchao 0.10, which this project doesn't use; peft >= 0.21 refuses to
# start LoRA when an old torchao is present ("only versions above 0.16.0 are supported").
!pip uninstall -q -y torchao
import torch
print('cuda', torch.cuda.is_available())


## 2. Check the connections

In [ ]:
!cd web && python manage.py check --database default && python manage.py showmigrations audit | tail -3
!cd web && python -c "import redis, os; print('redis ping:', redis.Redis.from_url(os.environ['REDIS_URL']).ping())"


## 3. Run the worker

Runs until you stop the cell. The batch page shows "worker online" for the Colab GPU while
it runs.

In [ ]:
!cd web && celery -A claimguard_web worker -Q gpu -c {CONCURRENCY} -n gpu@colab --loglevel=INFO
